# Core Moliterate concepts
This tutorial focuses on the core objects you will encounter when working with Moliterate: `ChemDataEntry`,
indexing semantics (relative, absolute, and origin indexes), and basic dataset exploration.


In [1]:
from pathlib import Path

import scm.moliterate as moliterate
from scm.moliterate import load_dataset

moliterate_path = Path(moliterate.__path__[0])
TUTORIAL_FOLDER = moliterate_path.parent.parent.parent / "tutorials"


## ChemDataEntry

A `ChemDataEntry` is the atomic unit of data in Moliterate. It bundles three things:

- **system**: an atomistic structure (ASE `Atoms` object)
- **properties**: a dictionary of computed values (energies, forces, etc.)
- **metadata**: JSON-compatible information (strings, numbers, lists, nested dicts)

It also carries **indexing fields** that record where the entry came from:
`idx_absolute` (position in the underlying dataset) and `idx_origin` (original identifier in the source).


In [2]:
from scm.moliterate.core.base_chem_dataset import BaseChemDataSet

ds: BaseChemDataSet = load_dataset(TUTORIAL_FOLDER / "data/params")
ds


ParAMSData(len=34, properties=['energy', 'forces'], transforms=[] at 0x760afe768040)

In [3]:
entry = ds[0]
entry


ChemDataEntry(COPt18, prop:['energy', 'forces'], md:['Frame', 'Origin', 'OriginalEnergyHartree', 'reference_engine', 'dataset'], idx_absolute=0, idx_origin=cleaned.job_frame001)

In [4]:
entry.chemical_system

ChemicalSystem(charge: 0.0, formula: COPt18, 3D-periodic)

In [5]:
list(entry.properties.keys())

['energy', 'forces']

In [6]:
list(entry.metadata.keys())

['Frame', 'Origin', 'OriginalEnergyHartree', 'reference_engine', 'dataset']

In [7]:
entry.idx_absolute, entry.idx_origin

(0, 'cleaned.job_frame001')

## Relative, absolute and origin indexes

Moliterate distinguishes three index concepts:

- **Relative index**: the position you use when indexing a dataset or subset (e.g. `ds[0]`).
- **Absolute index**: the row index in the *original* dataset backing the object.
- **Origin index**: the dataset-specific identifier for the entry (e.g. a job key in ParAMS).

When you create a subset, the *relative* indices are re-numbered starting at 0, but each row
still carries its original `idx_absolute` and `idx_origin` so you can trace it back.


In [8]:
for idx_relative, entry in enumerate(ds):
    entry.idx_absolute
    entry.idx_origin

In [9]:
subset = ds.subset([0, 2, 4])
subset

ParAMSData(len=3, properties=['energy', 'forces'], transforms=[] at 0x760aa499ea40)

In [10]:
subset.absolute_idxs


array([0, 2, 4])

In [11]:
subset[0].idx_absolute, subset[0].idx_origin


(0, 'cleaned.job_frame001')

In [12]:
# Convert absolute indices back to subset-relative positions
subset.from_absolute_to_relative(subset.select_indices())


array([0, 1, 2])

## Explore the dataset

Useful quick-inspection tools include:

- `__repr__` / `__str__`: a concise dataset summary

- `metadata`: global dataset metadata (source-dependent)

- `distance_unit`: units used for coordinates

- `out_properties`: properties *after* transforms (if any)

- `available_properties`: properties present in the raw dataset (there is also the properties_md_table, for formatting nicely the properties).

- sampling entries to inspect `system`, `properties`, and `metadata`


In [13]:
ds


ParAMSData(len=34, properties=['energy', 'forces'], transforms=[] at 0x760afe768040)

In [14]:
ds.metadata


{'General': {'dtype': 'JobCollection', 'version': '2025.206'},
 'PropUnitsShape': {'energy': ['Ha', 'float'], 'forces': ['Ha/bohr', (-1, 3)]},
 'jc_nEngines': 0,
 'all_settings': input: 	
       ams: 	
           Properties: 	
                      Gradients: 	yes
           Task: 	SinglePoint,
 'engine_collection': "---\ndtype: EngineCollection\nversion: '2025.206'\n...\n"}

In [15]:
ds.out_properties
# print(ds.properties_md_table())

[PropertyInfo(name='energy', unit='Ha', shape='float', description="Units might not be uniform! Found in ['training_set', 'validation_set']"),
 PropertyInfo(name='forces', unit='Ha/bohr', shape=(-1, 3), description="Units might not be uniform! Found in ['training_set', 'validation_set']")]

In [16]:
ds.distance_unit

'Ang'

In [17]:
# Inspect a few entries
[ds[i] for i in range(3)]


[ChemDataEntry(COPt18, prop:['energy', 'forces'], md:['Frame', 'Origin', 'OriginalEnergyHartree', 'reference_engine', 'dataset'], idx_absolute=0, idx_origin=cleaned.job_frame001),
 ChemDataEntry(COPt18, prop:['energy', 'forces'], md:['Frame', 'Origin', 'OriginalEnergyHartree', 'reference_engine', 'dataset'], idx_absolute=1, idx_origin=cleaned.job_frame002),
 ChemDataEntry(COPt18, prop:['energy', 'forces'], md:['Frame', 'Origin', 'OriginalEnergyHartree', 'reference_engine', 'dataset'], idx_absolute=2, idx_origin=cleaned.job_frame003)]

## Dataset writer

`BaseChemDataSet` is the **read-only** interface: it lets you iterate, slice, and inspect data.
`BaseChemDataSetWriter` extends that interface with **write capabilities** so you can create or modify datasets.

Key differences:

- **Creation**: writers provide a `create(...)` constructor to initialize an empty dataset.
- **Mutation**: writers implement `add_system(...)`, `add_systems(...)`, and metadata update methods.
- **Persistence**: writers typically map to a writable data source (file or in-memory).

Below is a minimal in-memory example using the `InMemoryMolData` writer.


In [18]:
from scm.moliterate.core.properties_info import PropertyInfo
from scm.moliterate.interfaces.in_memory import InMemoryMolData

writer = InMemoryMolData.create(
    available_properties=[PropertyInfo(name="energy", unit="hartree")],
    distance_unit="Ang",
)


In [19]:
# Add one entry from the existing dataset
writer.add_system(ds[0])
writer


InMemoryMolData(len=1, properties=['energy'], transforms=[] at 0x760aa4943bd0)

In [20]:
# Add one entry from the existing dataset
writer.add_systems(ds[:5])

100%|███████████████████████████████████████████████████████████████████████████████████████████████| 5/5 [00:00<00:00, 757.18it/s]


In [21]:
# Writers are still datasets, so you can read them like any BaseChemDataSet
writer[0]

ChemDataEntry(COPt18, prop:['energy'], md:['Frame', 'Origin', 'OriginalEnergyHartree', 'reference_engine', 'dataset'], idx_absolute=0, idx_origin=0)

In [22]:
# Batch collector: collect in memory and then flush into the dataset based on the batch_size
with writer.batch_collector(batch_size=10) as add:
    for entry_i in ds:
        add(entry_i)

100%|█████████████████████████████████████████████████████████████████████████████████████████████| 4/4 [00:00<00:00, 45839.39it/s]


In [23]:
writer

InMemoryMolData(len=40, properties=['energy'], transforms=[] at 0x760aa4943bd0)

In [24]:
# Note: you can disable the progress bar
from scm.moliterate.utils.progress_bar import MOLITERATE_PROGRESS_BAR_CONFIG

MOLITERATE_PROGRESS_BAR_CONFIG.disable = True

print(len(writer))
writer.add_systems(ds[:5])
print(len(writer))

40
45


## Entry Transform

Row transforms let you modify a single `ChemDataEntry` and returns the same `ChemDataEntry` modified.
Every transform is a `BaseEntryTransform` that implements three methods:

- `properties_in(...)`: which properties the transform expects as input

- `properties_out(...)`: how the properties schema changes after the transform

- `__call__(row)`: the actual per-row mutation or augmentation

Datasets keep a list of transforms in `ds.transforms`. When you access rows,
Moliterate applies the composed transform automatically and `ds.out_properties`
reflects the transformed schema.


In [25]:
ds.transforms

[]

In [26]:
ds.out_properties

[PropertyInfo(name='energy', unit='Ha', shape='float', description="Units might not be uniform! Found in ['training_set', 'validation_set']"),
 PropertyInfo(name='forces', unit='Ha/bohr', shape=(-1, 3), description="Units might not be uniform! Found in ['training_set', 'validation_set']")]

In [27]:
from scm.moliterate.transforms.atoms_stats import MinMaxAtomsDistance
from scm.moliterate.transforms.property_info_transform import ScalePropTransform

ds_trn = ds.subset()
ds_trn.transforms = [
    MinMaxAtomsDistance(min_out_key="min_dist", max_out_key="max_dist"),
    ScalePropTransform(key_in="energy", coeff_scale=27.2114, key_out="energy_ev"),
]

In [28]:
ds_trn.out_properties

[PropertyInfo(name='energy_ev', unit='Ha', shape='float', description="Units might not be uniform! Found in ['training_set', 'validation_set']"),
 PropertyInfo(name='forces', unit='Ha/bohr', shape=(-1, 3), description="Units might not be uniform! Found in ['training_set', 'validation_set']"),
 PropertyInfo(name='min_dist', unit=None, shape='float', description=''),
 PropertyInfo(name='max_dist', unit=None, shape='float', description='')]

In [29]:
ds_trn[0].properties

{'forces': array([[ 5.61223336e-05,  3.98221179e-05,  4.10494034e-05],
        [-1.79948688e-05, -1.91575945e-05, -3.94670461e-05],
        [ 6.83522210e-03, -3.89991984e-03,  2.63242840e-02],
        [-5.01019647e-03,  2.86435305e-03,  1.39489783e-02],
        [-2.11524013e-04, -1.26684267e-04,  1.45319326e-02],
        [-5.60599594e-06,  2.96235962e-04,  1.45346165e-02],
        [ 2.51261912e-05,  7.83574826e-03,  2.62984853e-02],
        [ 2.42179146e-04, -1.37341495e-04,  1.45055899e-02],
        [-6.84006900e-03, -3.93367616e-03,  2.63249974e-02],
        [ 4.97399620e-03,  2.86269295e-03,  1.39259669e-02],
        [-1.91170016e-05, -5.79514979e-03,  1.39405364e-02],
        [ 7.62320020e-04, -2.29540888e-04, -1.85812636e-02],
        [ 5.70425653e-04, -5.24644096e-04, -1.85750107e-02],
        [ 1.99037924e-04,  7.89070542e-04, -1.85918705e-02],
        [ 3.80067063e-05, -3.51206472e-05, -1.58442090e-02],
        [-5.77440781e-04, -5.61916181e-04, -1.85816012e-02],
        [-1.90

Note that the ScalePropTransform do not change the out_property units!

In [30]:
ds_trn_2 = ds.subset(transfer_transforms=False)
ds_trn_2.transforms.append(ScalePropTransform(key_in="energy", coeff_scale=100),)

ds_trn_2.out_properties


[PropertyInfo(name='energy', unit='Ha', shape='float', description="Units might not be uniform! Found in ['training_set', 'validation_set']"),
 PropertyInfo(name='forces', unit='Ha/bohr', shape=(-1, 3), description="Units might not be uniform! Found in ['training_set', 'validation_set']")]

In [31]:
ds_trn_2[0].properties


{'forces': array([[ 5.61223336e-05,  3.98221179e-05,  4.10494034e-05],
        [-1.79948688e-05, -1.91575945e-05, -3.94670461e-05],
        [ 6.83522210e-03, -3.89991984e-03,  2.63242840e-02],
        [-5.01019647e-03,  2.86435305e-03,  1.39489783e-02],
        [-2.11524013e-04, -1.26684267e-04,  1.45319326e-02],
        [-5.60599594e-06,  2.96235962e-04,  1.45346165e-02],
        [ 2.51261912e-05,  7.83574826e-03,  2.62984853e-02],
        [ 2.42179146e-04, -1.37341495e-04,  1.45055899e-02],
        [-6.84006900e-03, -3.93367616e-03,  2.63249974e-02],
        [ 4.97399620e-03,  2.86269295e-03,  1.39259669e-02],
        [-1.91170016e-05, -5.79514979e-03,  1.39405364e-02],
        [ 7.62320020e-04, -2.29540888e-04, -1.85812636e-02],
        [ 5.70425653e-04, -5.24644096e-04, -1.85750107e-02],
        [ 1.99037924e-04,  7.89070542e-04, -1.85918705e-02],
        [ 3.80067063e-05, -3.51206472e-05, -1.58442090e-02],
        [-5.77440781e-04, -5.61916181e-04, -1.85816012e-02],
        [-1.90

Some useful built-in `BaseRowTransform` implementations:

- `ToArrayTransform`: convert property values to NumPy arrays

- `ReshapeTransform`: reshape property arrays (supports `nAtoms`)

- `ConvertNamesTransform`: rename properties/metadata keys

- `ScalePropTransform`: scale a property (optionally rename)

- `MetadataAddTransform`: add or override row metadata

- `MinMaxAtomsDistance`: add min/max inter-atomic distances

- `PropertyTransform`: reduce or convert a property (e.g., norms, units)


## Nest Septs

Other tutorials in increasing order of complexity:

- __[advanced-features-filters.ipynb](advanced-features-filters.ipynb)__
- __[advanced-create-an-interface.ipynb](advanced-create-an-interface.ipynb)__